# WP2 — selected checkpoint, identified development export

This inference-only companion supplies Jonathan's missing WP2 records. It never trains, selects another checkpoint, reads official tests or changes historical files. **Default Run All does no model loading, image decoding, inference or export.** All processing is visible below; no project ML module is imported.

Use the existing WSL `as04_yolo_katana` kernel. The frozen checkpoint is ConvNeXt-Base, epoch1, authoritative224×224 preprocessing. Its legacy run plan also contains320×320: preserve that record and explicitly use the selected epoch's224resolution. This export is a new identified inference record, not a claim that historical anonymous rows have acquired identities.

Before any future export: save all cells, restart the kernel, verify NOTEBOOK_FILE points to this saved notebook, then explicitly enable RUN_EXPORT. The run retains one exact source snapshot and a separate code AST digest. Incomplete output directories are preserved if setup fails.


## 1. Explicit execution and paths

Save the notebook after editing configuration. Enable `RUN_EXPORT` only when ready to run development inference. Batch16 is a conservative starting configuration, not a measured GPU-memory guarantee. The historical checkpoint recorded BF16 and channels-last. Device, precision, batch size and backend settings are recorded; a new inference pass can differ slightly from historical metrics. Do not overwrite or reselect the historical checkpoint on that basis.

In [ ]:
from pathlib import Path
from contextlib import nullcontext
import ast,hashlib,json,platform,time,uuid
from datetime import datetime,timezone
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset,DataLoader
import torchvision
from torchvision.models import convnext_base
from torchvision.transforms import v2,InterpolationMode

def find_root(start=None):
    start=Path.cwd() if start is None else Path(start).resolve()
    for parent in (start,*start.parents):
        if (parent/'workspace/datasets/metadata/DPD_Small_train.csv').is_file():
            return parent
    raise FileNotFoundError('Open within the canonical WSL AS02 repository.')

ROOT=find_root()
RUN_EXPORT=False
DEVICE='cuda:0'
PRECISION='bf16'
BATCH_SIZE=16
NUM_WORKERS=0
CHANNELS_LAST=True
HANDOFF_DIR=ROOT/'workspace/handoffs/TEAM_WP5_HANDOFF_20261107/development_predictions'
CHECKPOINT=ROOT/'workspace/experiments/wp2/20261030T103349Z_multitask_convnext_base_22e3ef2f/selected_epoch_0001_3b01513b.pt'
CHECKPOINT_SHA256='7a0460b7d03bc08d19b68ca06cdd266746953322149a3edad177cf29992d7c04'
HANDOFF_SHA256='edbb1e97f0b520f867fddc88ddb1b26e27f5bd08704d0b72d55711c8b00ad777'
NOTEBOOK_FILE=ROOT/'workspace/handoffs/WP2_RELIABILITY_FIXES_20261107/source/notebooks/wp2_selected_development_export.ipynb'
OUTPUT_PARENT=ROOT/'workspace/experiments/wp2-identified-development-export'
print({'run_export':RUN_EXPORT,'root':str(ROOT),'torch':torch.__version__,
       'torchvision':torchvision.__version__,'batch_size':BATCH_SIZE,'device':DEVICE})


## 2. Read and validate only training-derived identities

The previously checksummed handoff pins all8000source records, the exact ordered4868/1238/1894folds and323non-contiguous global candidate IDs. Validate raw bytes, namespaced semantic hashes, every filename/truth/component mapping and fitting component coverage before decoding an image. No filename is attached to logits by guessing their order.

In [ ]:
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()

def fingerprint(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=True,allow_nan=False).encode()).hexdigest()

def checked_json(path,digest):
    if sha256_file(path)!=digest:
        raise ValueError(f'Raw SHA256 mismatch: {path}')
    return json.loads(Path(path).read_text())

def load_inputs(directory):
    index=checked_json(directory/'partial_handoff.json',HANDOFF_SHA256)
    manifest=checked_json(directory/index['identity_manifest']['path'],index['identity_manifest']['file_sha256'])
    if index['manifest_sha256']!='8e099e797491c235992a176a7f1261ae4e26f5370cde84095fc522a84bc93fac':
        raise ValueError('Unexpected identity manifest.')
    spec=manifest['spec'];candidates=spec['candidates'];samples=manifest['samples']
    source=directory/'source/DPD_Small_train.csv'
    if source.name!='DPD_Small_train.csv' or sha256_file(source)!=spec['source_sha256']:
        raise ValueError('Only the approved training-source bytes are accepted.')
    names=['image_name','crop_id','disease_id','composition_id','crop_name','disease_name','composition_name']
    frame=pd.read_csv(source,header=None,names=names)
    if len(frame)!=8000 or frame.isna().any().any() or frame.image_name.duplicated().any():
        raise ValueError('Invalid training-source rows.')
    ids=[r['composition_id'] for r in candidates];pairs=[[r['crop_id'],r['disease_id']] for r in candidates]
    if len(ids)!=323 or ids!=sorted(set(ids)) or len(set(map(tuple,pairs)))!=323:
        raise ValueError('Invalid323candidate ordering/pairs.')
    if fingerprint(candidates)!=spec['candidate_sha256'] or fingerprint([[i,*p] for i,p in zip(ids,pairs)])!='4cb0a24a96cf647f32311817fa7b082a46d3cf00c20ae61f4287e83cc898dfaf':
        raise ValueError('Candidate mapping or hash changed.')
    identity={r.image_name:r for r in frame.itertuples(index=False)}
    if len(samples)!=8000 or {s['sample_id'] for s in samples}!=set(identity):
        raise ValueError('Identities do not exactly cover training-source rows.')
    folds={fold:[] for fold in ['fitting','seen','unseen']}
    for s in samples:
        row=identity[s['sample_id']]
        if s['truth_id']!=int(row.composition_id) or s['fold'] not in folds:
            raise ValueError('Filename/truth/fold mismatch.')
        folds[s['fold']].append({k:(int(getattr(row,k)) if k.endswith('_id') else str(getattr(row,k))) for k in names})
    if [len(folds[f]) for f in folds]!=[4868,1238,1894]:
        raise ValueError('Fold denominator mismatch.')
    split={key:[r['image_name'] for r in folds[fold]] for key,fold in [('fit','fitting'),('val_seen','seen'),('val_unseen','unseen')]}
    if fingerprint(split)!='f0c7ba212449fd68f1d19c0f84cc91bf4cc1af29ae405128e5198bce89f25f85':
        raise ValueError('Ordered filename split changed.')
    fitting_ids={r['composition_id'] for r in folds['fitting']}
    if not {r['composition_id'] for r in folds['seen']}<=fitting_ids or fitting_ids&{r['composition_id'] for r in folds['unseen']}:
        raise ValueError('Fitting/validation composition separation changed.')
    mappings={task+'_id_to_name':{int(i):str(g[task+'_name'].iloc[0]) for i,g in frame.groupby(task+'_id')} for task in ['crop','disease']}
    for task,width in [('crop',55),('disease',175)]:
        if set(mappings[task+'_id_to_name'])!=set(range(width)) or {r[task+'_id'] for r in folds['fitting']}!=set(range(width)):
            raise ValueError('Component coverage changed.')
    for row in frame.itertuples(index=False):
        if '/' in row.image_name or chr(92) in row.image_name or ':' in row.image_name or row.image_name in ('.','..'):
            raise ValueError('Expected flat image filenames.')
    candidate_pairs={i:tuple(p) for i,p in zip(ids,pairs)}
    if any(candidate_pairs.get(int(r.composition_id))!=(int(r.crop_id),int(r.disease_id)) for r in frame.itertuples(index=False)):
        raise ValueError('Composition and component labels disagree.')
    return index,manifest,folds,mappings,ids,pairs

def capture_notebook_source(path,directory):
    """Retain/hash one saved byte snapshot; outputs do not affect the code digest.

    Save all cells and restart the kernel before an owner execution. This is
    saved-source provenance, not an attestation of unsaved/live kernel state.
    """
    raw=Path(path).read_bytes()
    notebook=json.loads(raw)
    code='\n'.join(''.join(cell['source']) for cell in notebook['cells'] if cell['cell_type']=='code')
    tree=ast.dump(ast.parse(code),include_attributes=False)
    snapshot=Path(directory)/'execution_source.ipynb'
    with snapshot.open('xb') as stream:stream.write(raw)
    return {'export_notebook_sha256':hashlib.sha256(raw).hexdigest(),
            'export_notebook_ast_sha256':hashlib.sha256(tree.encode('utf-8')).hexdigest(),
            'ast_hash_namespace':'python_ast_dump_without_attributes_v1',
            'source_snapshot':snapshot.name,'source_capture_scope':'saved_notebook_bytes_only'}


## 3. Visible selected-model reconstruction and preprocessing

No pretrained initialization is needed: the existing checkpoint contains the complete trained state. Torchvision is constructed with `weights=None`, retaining ConvNeXt pooling and final LayerNorm/flatten. Its original ImageNet linear layer is replaced; two1024→55/175heads use the exact state-dictionary names. Strict loading and complete class-mapping equality reject incompatible states. Legacy320run-plan metadata is acknowledged, never rewritten.

In [ ]:
def validate_selected_metadata(payload,mappings):
    expected={'format_version':1,'architecture':'multitask_convnext_base','num_crops':55,
              'num_diseases':175,'feature_dim':1024,'model_dtype':'torch.float32'}
    if any(payload.get(k)!=v for k,v in expected.items()):
        raise ValueError('Wrong checkpoint architecture, classes, feature dimension or dtype.')
    meta=payload.get('metadata',{});selection=meta.get('selection',{})
    if meta.get('epoch')!=1 or selection.get('epoch')!=1 or selection.get('resolution')!=224:
        raise ValueError('Require the frozen epoch1 checkpoint at its recorded224resolution.')
    if payload.get('class_mappings')!=mappings:
        raise ValueError('Checkpoint class mappings differ from original source IDs/names.')

class SelectedConvNeXtBase(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=convnext_base(weights=None)
        if self.backbone.classifier[-1].in_features!=1024:
            raise ValueError('Unexpected installed ConvNeXt-Base feature dimension.')
        self.backbone.classifier[-1]=nn.Identity()
        self.crop_head=nn.Linear(1024,55)
        self.disease_head=nn.Linear(1024,175)
    def forward(self,images):
        features=self.backbone(images)
        return {'crop_logits':self.crop_head(features),'disease_logits':self.disease_head(features),'features':features}

def load_selected_checkpoint(path,mappings,device):
    if sha256_file(path)!=CHECKPOINT_SHA256:
        raise ValueError('Selected checkpoint SHA256 mismatch.')
    payload=torch.load(path,map_location='cpu',weights_only=True)
    validate_selected_metadata(payload,mappings)
    model=SelectedConvNeXtBase()
    model.load_state_dict(payload['model_state_dict'],strict=True)
    model.eval().to(device)
    return model,{k:v for k,v in payload.items() if k!='model_state_dict'}

def validation_transform():
    return v2.Compose([v2.ToImage(),v2.Resize((224,224),interpolation=InterpolationMode.BILINEAR,antialias=True),
                       v2.ToDtype(torch.float32,scale=True),v2.Normalize(mean=[.485,.456,.406],std=[.229,.224,.225])])

class NamedDevelopmentImages(Dataset):
    def __init__(self,rows,image_directory):
        self.rows=list(rows);self.directory=Path(image_directory);self.transform=validation_transform()
    def __len__(self):return len(self.rows)
    def __getitem__(self,index):
        row=self.rows[index]
        with Image.open(self.directory/row['image_name']) as image:
            tensor=self.transform(image.convert('RGB'))
        return tensor,{'image_name':row['image_name'],'crop':row['crop_id'],
                       'disease':row['disease_id'],'composition':row['composition_id']}


## 4. Identity-carrying inference and canonical scores

Every batch includes original filenames and all three targets. Reject any missing, reordered or relabelled row. Snapshot CPU logits with `copy=True` **before the next forward pass**, including a final partial batch. No gradients, optimizer or selection occurs. The scorer retains the accepted PyTorch float64 log-softmax sum; ranking is stable in ascending original-ID columns, with no rounding or tolerance.

In [ ]:
def collect_named_logits(model,loader,rows,device,*,precision='bf16',channels_last=True):
    device=torch.device(device)
    if precision not in ['fp32','bf16']:
        raise ValueError('Use recorded BF16 onCUDA or explicit FP32 audit mode.')
    model.eval();crops=[];diseases=[];names=[];truths=[];offset=0
    with torch.inference_mode():
        for images,targets in loader:
            count=len(images);expected=rows[offset:offset+count]
            batch_names=list(targets['image_name'])
            if count<1 or batch_names!=[r['image_name'] for r in expected]:
                raise ValueError('Batch filenames are missing, duplicated or reordered.')
            for key,column in [('crop','crop_id'),('disease','disease_id'),('composition','composition_id')]:
                truth=targets[key].detach().cpu()
                if truth.dtype!=torch.int64 or not torch.equal(truth,torch.tensor([r[column] for r in expected],dtype=torch.int64)):
                    raise ValueError('Batch labels differ from identified source records.')
            images=images.to(device)
            if channels_last:images=images.contiguous(memory_format=torch.channels_last)
            amp=torch.autocast('cuda',dtype=torch.bfloat16) if device.type=='cuda' and precision=='bf16' else nullcontext()
            with amp:output=model(images)
            c=output['crop_logits'].detach().to(device='cpu',dtype=torch.float32,copy=True)
            d=output['disease_logits'].detach().to(device='cpu',dtype=torch.float32,copy=True)
            if c.shape!=(count,55) or d.shape!=(count,175) or not torch.isfinite(c).all() or not torch.isfinite(d).all():
                raise ValueError('Invalid/nonfinite component logits.')
            crops.append(c);diseases.append(d);names.extend(batch_names)
            truths.extend(targets['composition'].tolist());offset+=count
            if offset%128<count or offset==len(rows):print(f'Development inference:{offset}/{len(rows)}',flush=True)
    if offset!=len(rows) or not crops:raise ValueError('Incomplete or empty development inference.')
    return {'image_names':names,'ground_truth_ids':truths,'crop_logits':torch.cat(crops),'disease_logits':torch.cat(diseases)}

def score_components(crop_logits,disease_logits,candidate_ids,pairs):
    if not candidate_ids or candidate_ids!=sorted(set(candidate_ids)) or len(candidate_ids)!=len(pairs):
        raise ValueError('Candidate columns require unique ascending global IDs.')
    if len(set(map(tuple,pairs)))!=len(pairs) or any(len(p)!=2 or not 0<=p[0]<55 or not 0<=p[1]<175 for p in pairs):
        raise ValueError('Invalid candidate component pairs.')
    for logits,width in [(crop_logits,55),(disease_logits,175)]:
        if logits.ndim!=2 or logits.shape[0]<1 or logits.shape[1]!=width or not logits.is_floating_point() or not torch.isfinite(logits).all():
            raise ValueError('Invalid finite floating component logits.')
    if crop_logits.shape[0]!=disease_logits.shape[0] or crop_logits.device!=disease_logits.device:
        raise ValueError('Logit batch/device mismatch.')
    columns=torch.tensor(pairs,dtype=torch.int64,device=crop_logits.device)
    scores=F.log_softmax(crop_logits.to(torch.float64),dim=1)[:,columns[:,0]]+F.log_softmax(disease_logits.to(torch.float64),dim=1)[:,columns[:,1]]
    if not torch.isfinite(scores).all():raise ValueError('Nonfinite canonical composition scores.')
    return scores

def rank_global_ids(scores,ids):
    if scores.ndim!=2 or scores.shape[1]!=len(ids) or list(ids)!=sorted(set(ids)) or not torch.isfinite(scores).all():
        raise ValueError('Invalid score columns.')
    return torch.tensor(ids,dtype=torch.int64,device=scores.device)[torch.argsort(scores,dim=1,descending=True,stable=True)]

def create_output_directory(path):
    path=Path(path);path.mkdir(parents=True,exist_ok=False);return path


## 5. Manual export — disabled by default

This one guarded cell loads the selected checkpoint and runs **only** the two development folds. It never decodes fitting images or official-test images. BF16 requires a compatible CUDA device. Outputs go to a new unique run directory; failure leaves evidence without a completion receipt, never a resumable directory.

Canonical float64 score packets include independent crop/disease argmax IDs. A separate audit file retains the named component logits. WP5 owns metric reporting; no alternative scorer, Top-k accuracy, harmonic mean or checkpoint-selection implementation is added here. Reusing the four-model CLI still requires WP1 and the other available model packets to be assembled under an approved common envelope.

In [ ]:
if RUN_EXPORT:
    if type(BATCH_SIZE) is not int or BATCH_SIZE<1 or type(NUM_WORKERS) is not int or NUM_WORKERS<0:
        raise ValueError('Invalid loader settings.')
    device=torch.device(DEVICE)
    if device.type=='cuda' and (not torch.cuda.is_available() or PRECISION=='bf16' and not torch.cuda.is_bf16_supported()):
        raise RuntimeError('Requested CUDA/BF16 runtime unavailable; do not silently change precision.')
    directory=create_output_directory(OUTPUT_PARENT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:12]))
    source_identity=capture_notebook_source(NOTEBOOK_FILE,directory)
    index,manifest,folds,mappings,candidate_ids,pairs=load_inputs(HANDOFF_DIR)
    model,checkpoint_info=load_selected_checkpoint(CHECKPOINT,mappings,device)
    if CHANNELS_LAST:model=model.to(memory_format=torch.channels_last)
    metadata=checkpoint_info['metadata'];start=time.perf_counter()
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats(device)
    actual_preprocessing={'resolution':[224,224],'rgb':True,'validation_transform':repr(validation_transform()),
        'authority':'legacy metadata.selection.resolution=224; stored planned320 preserved unchanged',
        'precision':PRECISION if device.type=='cuda' else 'fp32','channels_last':CHANNELS_LAST}
    provenance={'model_id':'wp2_convnext_base','run_id':metadata['run_id'],'checkpoint_id':CHECKPOINT.name,
        'checkpoint_sha256':CHECKPOINT_SHA256,'preprocessing':json.dumps(actual_preprocessing,sort_keys=True),
        'code_version':json.dumps({**source_identity,'original_checkpoint_code':metadata['code']},sort_keys=True),
        'environment':json.dumps({'python':platform.python_version(),'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),
            'device':str(device),'cuda_build':torch.version.cuda,'batch_size':BATCH_SIZE,'num_workers':NUM_WORKERS,
            'cudnn_benchmark':torch.backends.cudnn.benchmark,'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),
            'cuda_matmul_tf32':torch.backends.cuda.matmul.allow_tf32,'cudnn_tf32':torch.backends.cudnn.allow_tf32},sort_keys=True),
        'score_semantics':'sum_log_softmax_v1','score_precision':'float64',
        'comparison_budget':'Frozen-finalist development-323 reporting; heterogeneous original training budgets disclosed, not a matched-training-budget claim',
        'prompt_sha256':'','metric_policy':'composition_hits_fraction_min5_v1'}
    serialized={};counts={}
    for fold in ['seen','unseen']:
        loader=DataLoader(NamedDevelopmentImages(folds[fold],ROOT/'workspace/datasets/DPD_Small_dataset'),
            batch_size=BATCH_SIZE,shuffle=False,drop_last=False,num_workers=NUM_WORKERS)
        result=collect_named_logits(model,loader,folds[fold],device,precision=PRECISION,channels_last=CHANNELS_LAST)
        scores=score_components(result['crop_logits'],result['disease_logits'],candidate_ids,pairs)
        ranks=rank_global_ids(scores,candidate_ids)
        packet={'schema':'wp5-model-output-v1','model_kind':'wp2_convnext_base',**provenance,
            'manifest_sha256':index['manifest_sha256'],'source_sha256':manifest['spec']['source_sha256'],
            'candidate_ids':candidate_ids,'candidate_sha256':'4cb0a24a96cf647f32311817fa7b082a46d3cf00c20ae61f4287e83cc898dfaf',
            'candidate_hash_namespace':'wp2-json-global-id-crop-disease-triples-v1',
            'split_sha256':'f0c7ba212449fd68f1d19c0f84cc91bf4cc1af29ae405128e5198bce89f25f85',
            'split_hash_namespace':'wp2-json-fit-val_seen-val_unseen-filenames-v1',
            'selection':{'architecture':'multitask_convnext_base','epoch':1,'input_resolution':224},
            'fold':fold,'image_names':result['image_names'],'ground_truth_ids':result['ground_truth_ids'],
            'scores':scores.tolist(),'independent_crop_ids':torch.argmax(result['crop_logits'],1).tolist(),
            'independent_disease_ids':torch.argmax(result['disease_logits'],1).tolist()}
        serialized[fold+'_packet.json']=json.dumps(packet,allow_nan=False)+'\n'
        audit={k:(value.tolist() if isinstance(value,torch.Tensor) else value) for k,value in result.items()}
        audit['ranked_global_ids_top5']=ranks[:,:5].tolist()
        serialized[fold+'_component_audit.json']=json.dumps(audit,allow_nan=False)+'\n'
        counts[fold]=len(result['image_names'])
    if counts!={'seen':1238,'unseen':1894}:raise ValueError('Incomplete folds; no successful export.')
    snapshot=directory/source_identity['source_snapshot']
    ledger=[{'path':snapshot.name,'file_sha256':source_identity['export_notebook_sha256'],'bytes':snapshot.stat().st_size}]
    for name,text in serialized.items():
        path=directory/name
        with path.open('x',encoding='utf-8') as stream:stream.write(text)
        ledger.append({'path':name,'file_sha256':sha256_file(path),'bytes':path.stat().st_size})
    receipt={'schema':'wp2-identified-development-export-v1','model_provenance':provenance,'selection':packet['selection'],
        'manifest_sha256':index['manifest_sha256'],'identity_manifest_raw_sha256':index['identity_manifest']['file_sha256'],
        'counts':counts,'files':ledger,'elapsed_seconds':time.perf_counter()-start,
        'peak_cuda_allocated_bytes':torch.cuda.max_memory_allocated(device) if device.type=='cuda' else None,
        'historical_checkpoint_selection_preserved':metadata['selection'],'historical_raw_preprocessing':checkpoint_info['preprocessing'],
        'historical_reselection_performed':False,'official_tests_used':False,'training_performed':False}
    with (directory/'completion.json').open('x',encoding='utf-8') as stream:json.dump(receipt,stream,indent=2,allow_nan=False)
    print({'completed_export':str(directory),'counts':counts,'files':ledger})
else:
    print('RUN_EXPORT=False: no checkpoint loading, image decoding, inference or output writes. Enable only for an owner-executed development export.')


## 6. After execution

Keep the completed run unchanged. Supply `completion.json`, the two score packets and optional component audits to Chia/Jonathan; their SHA256values are in the receipt. Validate the packets through PR18's `ModelOutputAdapter` against the existing manifest before inclusion in a four-model report. Any metric difference from the old selected checkpoint summary is an inference-provenance investigation, not permission to tune, reselect or rewrite history.

This notebook does not fix the historical diagnostic collector in PR5 or modify its publication branch. Its new collector snapshots outputs safely and carries identities. The separate two-line PR5 correction remains coordinated with Jonathan. No new model result exists until the owner executes this notebook.